In [ ]:
# Problema: Convertir un extracto real de la colección de un museo en datos relacionales que conserven obras, creadores y citas.

import sqlite3
from pathlib import Path

import pandas as pd

ROOT = next(path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / 'data').is_dir() and (path / 'submission').is_dir())
SOURCE = ROOT / 'data' / 'museum_collection.csv'
OUTPUT = ROOT / 'submission' / 'museum_collection.db'


In [ ]:
# Cada fila es una referencia bibliográfica de una obra, no necesariamente una obra distinta.
raw = pd.read_csv(SOURCE)
raw.shape, raw['itemid'].nunique(), raw['citation'].nunique()


In [ ]:
# Una obra queda una sola vez. Los atributos ausentes se conservan como nulos: no se inventan datos.
work_metadata = raw.dropna(subset=['title']).sort_values('itemid').drop_duplicates('itemid')
categories = work_metadata[['category']].dropna().drop_duplicates().sort_values('category').reset_index(drop=True)
categories['category_id'] = categories.index + 1
artworks = (raw[['itemid']].drop_duplicates().rename(columns={'itemid': 'artwork_id'})
    .merge(work_metadata[['itemid', 'title', 'creation_date', 'category']], left_on='artwork_id', right_on='itemid', how='left')
    .merge(categories, on='category', how='left')[['artwork_id', 'title', 'creation_date', 'category_id']])
creators = (raw.dropna(subset=['creatorid'])[['creatorid', 'creator', 'birth_year', 'death_year']]
    .drop_duplicates('creatorid').rename(columns={'creatorid': 'creator_id'}))
artwork_creators = (raw.dropna(subset=['creatorid'])[['itemid', 'creatorid']].drop_duplicates()
    .rename(columns={'itemid': 'artwork_id', 'creatorid': 'creator_id'}))
citations = (raw.dropna(subset=['citation'])[['itemid', 'citation']].drop_duplicates()
    .rename(columns={'itemid': 'artwork_id'}).reset_index(drop=True))
citations.insert(0, 'citation_id', citations.index + 1)
artworks.shape, creators.shape, artwork_creators.shape, citations.shape


In [ ]:
# Las claves foráneas hacen explícitas las relaciones que el archivo plano escondía.
OUTPUT.unlink(missing_ok=True)
with sqlite3.connect(OUTPUT) as connection:
    connection.execute('PRAGMA foreign_keys = ON')
    connection.executescript('''
        CREATE TABLE categories (category_id INTEGER PRIMARY KEY, category TEXT NOT NULL UNIQUE);
        CREATE TABLE artworks (artwork_id INTEGER PRIMARY KEY, title TEXT, creation_date TEXT, category_id INTEGER, FOREIGN KEY (category_id) REFERENCES categories(category_id));
        CREATE TABLE creators (creator_id INTEGER PRIMARY KEY, creator TEXT, birth_year REAL, death_year REAL);
        CREATE TABLE artwork_creators (artwork_id INTEGER, creator_id INTEGER, PRIMARY KEY (artwork_id, creator_id), FOREIGN KEY (artwork_id) REFERENCES artworks(artwork_id), FOREIGN KEY (creator_id) REFERENCES creators(creator_id));
        CREATE TABLE citations (citation_id INTEGER PRIMARY KEY, artwork_id INTEGER NOT NULL, citation TEXT NOT NULL, FOREIGN KEY (artwork_id) REFERENCES artworks(artwork_id));
    ''')
    categories[['category_id', 'category']].to_sql('categories', connection, if_exists='append', index=False)
    artworks.to_sql('artworks', connection, if_exists='append', index=False)
    creators.to_sql('creators', connection, if_exists='append', index=False)
    artwork_creators.to_sql('artwork_creators', connection, if_exists='append', index=False)
    citations.to_sql('citations', connection, if_exists='append', index=False)


In [ ]:
# Esta consulta devuelve la vista analítica sin duplicar la información en el almacenamiento.
with sqlite3.connect(OUTPUT) as connection:
    summary = pd.read_sql_query('''
        SELECT c.category, COUNT(DISTINCT a.artwork_id) AS artworks, COUNT(DISTINCT ac.creator_id) AS creators
        FROM categories c JOIN artworks a USING(category_id) LEFT JOIN artwork_creators ac USING(artwork_id)
        GROUP BY c.category ORDER BY artworks DESC, c.category
    ''', connection)
summary
